In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime,UTC
import uuid



In [0]:
spark.sql("use catalog novacart")
spark.sql("create schema if not exists gold")
gold_run_id=uuid.uuid4()
run_ts_str=datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
run_date_str=datetime.now(UTC).strftime("%Y-%m-%d")

print("current gold_run_id=",gold_run_id)
print("current time stamp",run_ts_str)
print("current date",run_date_str)

In [0]:
spark.sql("""
          create table if not exists novacart.gold.processing_control(
              layer string,
              entity_name string,
              last_processed_silver_run_id string,
              last_processed_silver_run_ts timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp


          )
          using delta
          """)

In [0]:
def upsert_to_gold(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)
         

        

In [0]:
def get_last_processed_silver_ts(entity_name):
    ctrl=(
        spark.table("novacart.gold.processing_control")
        .filter((col("layer")=="gold")&
                (col("entity_name")==entity_name)&
                (col("run_status")=="success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )

    rows=ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name,last_processed_silver_run_id,last_processed_run_ts,rows_merged):
    ctrl_df=spark.createDataFrame(
    [(
        "gold",entity_name,last_processed_silver_run_id,last_processed_run_ts,int(rows_merged),"success",gold_run_id,datetime.now(UTC)
    )],
    schema="""
    layer string,
    entity_name string,
    last_processed_silver_run_id string,
    last_processed_silver_run_ts timestamp,
    rows_merged bigint,
    run_status string,
    gold_run_id string,
    updated_at timestamp
    """       
    )

    dt=DeltaTable.forName(spark,"novacart.gold.processing_control")
    (dt.alias("t")
     .merge(ctrl_df.alias("s"),"t.layer = s.layer and t.entity_name=s.entity_name")
     .whenMatchedUpdate(set={"last_processed_silver_run_id":"s.last_processed_silver_run_id",
                             "last_processed_silver_run_ts":"s.last_processed_silver_run_ts",
                             "rows_merged":"s.rows_merged",
                             "run_status":"s.run_status",
                             "gold_run_id":"s.gold_run_id",
                             "updated_at":"s.updated_at"})
     .whenNotMatchedInsertAll()
     .execute()
    )

In [0]:
last_gold_ts=get_last_processed_silver_ts("orders_information")

print("last processed silver timestamp for gold",last_gold_ts)
silver_orders_current=spark.read.table("novacart.silver.orders_transformed")
silver_products_current=spark.read.table("novacart.silver.products_transformed")
silver_payments_current=spark.read.table("novacart.silver.payments_transformed")

if last_gold_ts is None:
    changed_orders=silver_orders_current 
    changed_products=silver_products_current
    changed_payments=silver_payments_current
else:
    changed_orders=silver_orders_current.filter(col("updated_at")>lit(last_gold_ts))
    changed_products=silver_products_current.filter(col("updated_at")>lit(last_gold_ts))
    changed_payments=silver_payments_current.filter(col("updated_at")>lit(last_gold_ts))

changed_orders_count=changed_orders.count()
changed_products_count=changed_products.count()
changed_payments_count=changed_payments.count()

print(f"number of changed orders={changed_orders_count}")
print(f"number of changed products={changed_products_count}")
print(f"number of changed payments={changed_payments_count}")


In [0]:
impacted_from_orders=changed_orders.select("order_id").distinct()
impacted_from_payments=changed_payments.select("order_id").distinct()
impacted_from_products=(
    changed_products.alias("p").join(silver_orders_current.alias("o"),col("p.product_id")==col("o.product_id"),"inner")    
.select(col("o.order_id")).distinct())

impacted_order_ids=(
    impacted_from_orders.union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)
print("impacted_order_ids=",impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))



In [0]:
impacted_order = (
 silver_orders_current.alias("o").join(impacted_order_ids.alias("i"),col("o.order_id")==col("i.order_id"),"inner").select("o.*")
)
# impacted_products=(
#     silver_products_current.alias("p").join(impacted_order_ids.alias("i"),col("p.product_id")==col("i.order_id"),"inner")
# )
# impacted_payments=(
#     silver_payments_current.alias("p").join(impacted_order_ids.alias("i"),col("p.order_id")==col("i.order_id"),"inner")
# )
# (
#     impacted_order.alias("o").join(impacted_products.alias("p"),col("o.order_id")==col("p.product_id"),"inner")
#     .join(impacted_payments.alias("pay"),col("o.order_id")==col("pay.order_id"),"inner")

# )
gold_delta = (
    impacted_order.alias("o")
    .join(
        silver_products_current.alias("p"),
        col("o.product_id") == col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        col("o.order_id") == col("py.order_id"),
        "inner"
    )
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("p.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.price").alias("product_price"),
        col("o.order_status"),
        col("o.order_amount"),
        col("py.payment_id"),
        col("py.payment_status"),
        col("py.paid_amount"),
        col("o.order_date"),
        col("o.order_month"),
        col("o.order_year"),
        greatest(
            col("o.updated_at").cast("timestamp"),
            col("p.updated_at").cast("timestamp"),
            col("py.bronze_ingested_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        when(
            col("order_amount") > 0,
            col("paid_amount") / col("order_amount")
        ).otherwise(lit(0))
    )
    .withColumn(
        "payment_state",
        when(
            col("order_amount") == 0,
            lit("invalid_order_amount")
        )
        .when(col("payment_completion_ratio")==0,"unpaid")
        .when(col("payment_completion_ratio")==1,"paid")
        .when(col("payment_completion_ratio")<1,"partially_paid")
        .when(col("payment_completion_ratio")>1,"overpaid")
    )
    .withColumn("gold_updated_date",to_date(col("gold_update_ts")))
    .withColumn("gold_run_id",lit(str(gold_run_id)))
)
print("gold_delta_rows",gold_delta.count())
#display(gold_delta)

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
    


In [0]:
if gold_delta.count()>0:
    upsert_to_gold(gold_delta,"novacart.gold.order_information","order_id")
else:
    print("no new records to insert into gold table")


In [0]:
if not spark.catalog.tableExists("novacart.gold.order_information_scd2"):
    spark.sql("""
              create table novacart.gold.order_information_scd2 
              using delta
              as 
              select *,cast(null as timestamp) as valid_from_ts,cast(null as timestamp) as valid_to_ts,
              true as is_current from novacart.gold.order_information where 1=0
              """)
if gold_delta.count()>0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
merge into novacart.gold.order_information_scd2 t using gold_delta_view s 
 on t.order_id=s.order_id  and t.is_current=true
when matched and(
    not (t.order_status <=> s.order_status)or
    not (t.order_amount<=>s.order_amount) or
    not (t.paid_amount<=>s.paid_amount) or
    not (t.payment_id<=>s.payment_id) or
    not (t.category<=>s.category) or
    not (t.product_name<=>s.product_name) or
    not (t.product_price<=>s.product_price))

    then update set 
    is_current=false,
    valid_to_ts=s.gold_update_ts
""")    
spark.sql("""
          insert into novacart.gold.order_information_scd2
          select s.*,s.gold_update_ts as valid_from_ts,cast(null as timestamp) as valid_to_ts,
          true as is_current
          from gold_delta_view s 
          left join novacart.gold.order_information_scd2 t 
          on s.order_id=t.order_id and t.is_current=true
          where t.order_id is null or (
              not (t.order_status <=> s.order_status) or 
              not (t.order_amount<=> s.order_amount) or 
              not (t.paid_amount <=> s.paid_amount) or 
              not (t.payment_id <=> s.payment_id ) or 
              not (t.category <=> s.category) or 
              not (t.product_name<=> s.product_name) or 
              not (t.product_price <=> s.product_price))
          

          
          """)
              

In [0]:
if gold_delta.count()>0:
    impacted_categories=(
        gold_delta.select("category")
        .filter(col("category").isNotNull())
        .distinct()
              
    )
    category_perf_delta=(
        spark.read.table("novacart.gold.order_information")
        .join(impacted_categories,"category","inner")
        .groupBy("category")
        .agg(countDistinct("order_id").alias("total_products"),
             sum(when(col("order_amount")>0,col("order_amount")).otherwise(lit(0.0))).alias("gross_merchandise_value"),
             sum(when(col("paid_amount")>0,col("paid_amount")).otherwise(lit(0.0))).alias("total_paid_amount"),
             avg(col("payment_completion_ratio")).alias("average_payment_completion_raio"),
             (sum(when(col("payment_status")=="FAILED",1).otherwise(0))/count("*")).alias("payment_failure_rate") 
                
             )
    )
    upsert_to_gold(category_perf_delta,"novacart.gold.category_performance","category")


In [0]:
%sql
select * from novacart.gold.category_performance

In [0]:
spark.sql("create volume if not exists novacart.gold.gold_snapshots_vol")

In [0]:
latest_order_path=(
"/Volumes/novacart/gold/gold_snapshots_vol/gold_latest/order_information"
)

latest_category_path=(
"/Volumes/novacart/gold/gold_snapshots_vol/gold_latest/category_performace")
 
historical_orders_path=f"/Volumes/novacart/gold/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"

historical_category_path=f"/Volumes/novacart/gold/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"
spark.read.table("novacart.gold.order_information").write.mode("overwrite").format("parquet").save(latest_order_path)
spark.read.table("novacart.gold.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("novacart.gold.order_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("novacart.gold.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("latest orders path:",latest_order_path)
print("latest categories path:",latest_category_path)
print("historical orders path:",historical_orders_path)
print("historical category path:",historical_category_path)

In [0]:
latest_silver_ts=silver_orders_current.agg(max("bronze_ingested_at").alias("mx")).collect()[0]['mx']
latest_silver_run_id=(

    silver_orders_current
    .filter(col("bronze_ingested_at")==latest_silver_ts)
    .agg(max("silver_run_id").alias("mx"))
    .collect()[0]['mx']
)if latest_silver_ts is not None else None    
upsert_gold_control("orders_information",latest_silver_run_id,latest_silver_ts,gold_delta.count())
display(spark.table("novacart.gold.processing_control"))

